# Stemness scoring shared by Figures S2 and S3

Calculate the predefined stemness score once in the same six-state epithelial population used by the original Fig. S3 scoring code. Export cell barcodes, cell_subtype, and stemness_score to the original analysis directory. The four-kernel analysis in figs2-3_velovi_cellrank_four_kernels.ipynb and the Fig. S3 displays both read this table; neither recalculates the score.

Run this notebook before figs2-3_velovi_cellrank_four_kernels.ipynb and figs3/figs3-1_stemness_and_epithelial_survival.ipynb. Existing embeddings and scores unrelated to stemness are not recalculated here.


In [ ]:
from pathlib import Path

import numpy as np
import scanpy as sc

BASE_DIR = Path('/mnt/disk18t/lr_xcy/riku/kirc_205/leiden_detailed/dpt_root_from_miox')
H5AD_PATH = BASE_DIR / 'adata_epi_dpt_rootmiox.h5ad'
STEMNESS_SCORES_PATH = BASE_DIR / 'stemness_scores_six_states.tsv'


In [ ]:
adata = sc.read_h5ad(H5AD_PATH)
if adata.raw is None:
    raise ValueError("The stemness scoring input must contain the normalized full-gene raw matrix.")
print(adata)


In [ ]:
# Recalculate stemness scores while retaining the saved embedding coordinates.
excluded_states = ["Epi_CD44_high", "Epi_AQP2", "Epi_CA12"]
adata = adata[~adata.obs["cell_subtype"].isin(excluded_states)].copy()

STEMNESS_GENES = [
    "ACLY", "ADAM17", "AKT1", "ALDH1B1", "ALDH1L1", "BMI1", "CA9", "CD24",
    "CD34", "CD44", "CDKN1A", "CDKN2A", "CTBP2", "CXCR4", "DHX9", "DOT1L",
    "E2F2", "EGF", "EPAS1", "EZH2", "FKBP5", "FZD4", "GJA1", "GLI1", "HIF1A",
    "HOTAIR", "ID4", "IGF1R", "IL22", "ITGA6", "JAK2", "KDR", "KLF4", "KLF5",
    "KLF9", "L1TD1", "LGR5", "LILRB2", "LIN28A", "MCAM", "MICU1", "NANOG",
    "NCAM1", "NES", "NOTCH1", "PHLDA2", "PLAU", "POU5F1", "PROM1", "PSMD10",
    "PTEN", "PTTG1", "RAC1", "RECK", "REG4", "S100A4", "SCGB2A1", "SEMA3F",
    "SIRT2", "SMAD6", "SOX2", "SPDYA", "STAT3", "TERT", "TGFB1", "TGFBR2",
    "TP53", "TP63", "TWIST1", "USP7", "YAP1", "YBX1",
]
raw_genes = set(adata.raw.var_names)
stemness_genes_present = [gene for gene in STEMNESS_GENES if gene in raw_genes]
sc.tl.score_genes(
    adata,
    gene_list=stemness_genes_present,
    score_name="stemness_score",
    use_raw=True,
)


In [ ]:
# Export the shared scores for four-kernel root selection and Fig. S3 displays.
stemness_export = adata.obs.loc[:, ["cell_subtype", "stemness_score"]].copy()
if not stemness_export.index.is_unique:
    raise ValueError("Stemness export requires unique cell barcodes.")
if not np.isfinite(stemness_export["stemness_score"].to_numpy(dtype=float)).all():
    raise ValueError("Stemness export contains missing or non-finite scores.")
stemness_export.to_csv(
    STEMNESS_SCORES_PATH,
    sep="\t",
    index=True,
    index_label="cell_barcode",
    float_format="%.17g",
)
print("exported stemness scores:", STEMNESS_SCORES_PATH, "cells:", len(stemness_export))
